# Sector Normalization

This notebook normalizes industry sectors and classifications to standard taxonomies including NAICS (North American Industry Classification System), GICS (Global Industry Classification Standard), and custom taxonomies.

## Architecture

**Input**: `workspace.silver.silver_jobs_current`  
**Output**: `workspace.semantic.sem_sector_map`  
**Review Queue**: `workspace.silver.silver_semantic_review_queue`  
**Mode**: Incremental (process new batches only)

## Batch Processing

- Tracks processed batches in `metadata.sector_normalization_batches`
- Automatically processes all unprocessed batches
- Idempotent: safe to re-run
- Confidence scoring at each stage

## Standard Taxonomies
1. **NAICS**: 2-6 digit hierarchical codes (e.g., 541511 = Custom Computer Programming Services)
2. **GICS**: 8-digit codes organized in 4 levels (Sector, Industry Group, Industry, Sub-Industry)
3. **SIC**: Standard Industrial Classification (legacy)
4. **Custom**: Organization-specific taxonomies

In [0]:
dbutils.widgets.text("batch_id", "", "Batch ID (leave empty for all unprocessed)")

batch_id = dbutils.widgets.get("batch_id").strip()

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, FloatType, IntegerType, ArrayType, TimestampType
from pyspark.sql.window import Window
from datetime import datetime
import re

CATALOG = "workspace"
SILVER_SCHEMA = f"{CATALOG}.silver"
SEMANTIC_SCHEMA = f"{CATALOG}.semantic"
METADATA_SCHEMA = f"{CATALOG}.metadata"

# Configuration
CONFIG = {
    "input_table": f"{SILVER_SCHEMA}.silver_jobs_current",
    "output_table": f"{SEMANTIC_SCHEMA}.sem_sector_map",
    "review_queue_table": f"{SILVER_SCHEMA}.silver_semantic_review_queue",
    "primary_taxonomy": "NAICS",  # Primary taxonomy to use
    "confidence_threshold": 0.8,
    "enable_hierarchical_rollup": True,  # Roll up to parent categories if needed
    "max_taxonomy_level": 4
}

run_id = datetime.now().strftime("%Y%m%d_%H%M%S")
run_timestamp_py = datetime.now()  # Python timestamp for metadata records
run_timestamp = F.current_timestamp()  # Spark timestamp for DataFrame operations

In [0]:
# Create metadata table to track sector-normalized batches
metadata_table = f"{METADATA_SCHEMA}.sector_normalization_batches"

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {metadata_table} (
  batch_id STRING,
  source_name STRING,
  sectors_processed INT,
  canonical_count INT,
  review_queue_count INT,
  processed_at TIMESTAMP,
  run_id STRING,
  status STRING
)
USING DELTA
COMMENT 'Tracks which batches have been sector-normalized'
""")

# Define metadata schema for recording batch processing
metadata_schema = StructType([
    StructField("batch_id", StringType(), True),
    StructField("source_name", StringType(), True),
    StructField("sectors_processed", IntegerType(), True),
    StructField("canonical_count", IntegerType(), True),
    StructField("review_queue_count", IntegerType(), True),
    StructField("processed_at", TimestampType(), True),
    StructField("run_id", StringType(), True),
    StructField("status", StringType(), True)
])

In [0]:
# ⚠️ UNCOMMENT TO RESET: Deletes all sector mappings and metadata to reprocess everything
# Use this when taxonomy/logic changes require full reprocessing

# print("⚠️  RESETTING: Deleting all sector mappings and metadata...")
# spark.sql(f"DELETE FROM {CONFIG['output_table']}")
# spark.sql(f"DELETE FROM {metadata_table}")
# spark.sql(f"DELETE FROM {CONFIG['review_queue_table']} WHERE issue_type IN ('low_confidence', 'no_match')")
# print("✅ Reset complete. All batches will be reprocessed with updated taxonomy.")

In [0]:
# Identify unprocessed batches
if batch_id:
    # Process specific batch if provided
    batches_to_process = [(batch_id, None)]  # (batch_id, source_name) tuple
    print(f"Processing specific batch: {batch_id}")
else:
    # Find all batches in current table
    all_batches_df = spark.table(CONFIG["input_table"]).select(
        F.col("current_batch_id").alias("batch_id"),
        F.col("source_name")
    ).distinct()
    
    # Find already processed batches
    processed_batches_df = spark.table(metadata_table).select(
        F.col("batch_id"),
        F.col("source_name")
    ).distinct()
    
    # Get unprocessed batches
    unprocessed_df = all_batches_df.join(
        processed_batches_df,
        ["batch_id", "source_name"],
        "left_anti"
    )
    
    batches_to_process = [(row.batch_id, row.source_name) for row in unprocessed_df.collect()]
    print(f"Found {len(batches_to_process)} unprocessed batch(es) to process")

In [0]:
# Load sector taxonomy from governed metadata files
# This replaces hardcoded tech-centric NAICS samples with sector-agnostic taxonomy

METADATA_BASE = "/Workspace/Users/aaryan.shrivastav1403@gmail.com/LMIP/metadata"

print("Loading sector taxonomy from metadata...")
sectors_raw_df = spark.read.csv(
    f"{METADATA_BASE}/sectors.csv",
    header=True,
    inferSchema=True
)

# Transform to match existing schema expectations
# Map metadata structure to NAICS-compatible structure for backward compatibility
sector_taxonomy_df = sectors_raw_df.select(
    F.coalesce(F.col("naics_code"), F.col("sector_key")).alias("sector_code"),
    F.col("sector_name"),
    F.lit("NAICS").alias("taxonomy_type"),
    F.when(F.col("parent_sector").isNull(), 1).otherwise(2).alias("level"),
    F.col("parent_sector").alias("parent_code"),
    F.split(F.col("keywords"), "\\|").alias("keywords")
)


In [0]:
# Load sector assignments from unprocessed batches only
if len(batches_to_process) == 0:
    print("\n✓ No unprocessed batches found. All sectors are already normalized.")
    dbutils.notebook.exit('{"status": "no_batches", "message": "All batches already processed"}')

print(f"\nLoading sector assignments from {len(batches_to_process)} unprocessed batch(es)...")

# Get batch IDs to process
batch_ids_to_process = [batch_id for batch_id, _ in batches_to_process]

jobs_df = spark.table(CONFIG["input_table"])

extracted_sectors_df = jobs_df.filter(
    F.col("current_batch_id").isin(batch_ids_to_process)
).select(
    F.col("enterprise_job_id").alias("extraction_id"),
    F.col("sector_assigned").alias("extracted_sector"),
    F.col("sector_assignment_method").alias("assignment_method"),
    F.col("sector_confidence").alias("input_confidence"),
    F.col("enterprise_job_id").alias("source_id"),
    F.col("current_batch_id").alias("batch_id"),
    F.col("source_name")
).filter(
    F.col("sector_assigned").isNotNull()
)

extracted_count = extracted_sectors_df.count()
print(f"✓ Loaded {extracted_count} sector assignments from {len(batches_to_process)} batch(es)")

if extracted_count == 0:
    print("\n⚠ No sector assignments in these batches.")
    dbutils.notebook.exit('{"status": "no_sectors", "message": "No sectors found in unprocessed batches"}')

display(extracted_sectors_df.limit(10))

In [0]:
def normalize_sector_name(name):
    """Normalize sector name for matching."""
    if not name:
        return ""
    
    # Convert to lowercase
    name = name.lower().strip()
    
    # Remove common words
    stop_words = ["and", "the", "of", "in", "for", "services", "industry"]
    words = name.split()
    words = [w for w in words if w not in stop_words]
    
    return " ".join(words)

from pyspark.sql.functions import udf
from pyspark.sql.types import StringType

normalize_udf = udf(normalize_sector_name, StringType())

# Normalize extracted sectors
extracted_normalized_df = extracted_sectors_df.withColumn(
    "normalized_sector",
    normalize_udf(F.col("extracted_sector"))
)

print("Normalized extracted sectors:")
display(extracted_normalized_df)

In [0]:
from pyspark.sql.functions import explode, array

# Explode taxonomy keywords for matching
taxonomy_keywords_df = sector_taxonomy_df.select(
    "sector_code",
    "sector_name",
    "taxonomy_type",
    "level",
    "parent_code",
    explode("keywords").alias("keyword")
).withColumn(
    "normalized_keyword",
    normalize_udf(F.col("keyword"))
)

print(f"Expanded taxonomy to {taxonomy_keywords_df.count()} keyword entries")
display(taxonomy_keywords_df.limit(20))

In [0]:
# Match extracted sectors against taxonomy keywords
exact_matches_df = extracted_normalized_df.join(
    taxonomy_keywords_df,
    extracted_normalized_df.normalized_sector == taxonomy_keywords_df.normalized_keyword,
    "left"
).select(
    extracted_normalized_df.extraction_id,
    extracted_normalized_df.extracted_sector,
    extracted_normalized_df.source_id,
    extracted_normalized_df.normalized_sector,
    extracted_normalized_df.batch_id,
    extracted_normalized_df.source_name,
    F.col("sector_code"),
    F.col("sector_name"),
    F.col("taxonomy_type"),
    F.col("level"),
    F.col("parent_code"),
    F.lit("exact_keyword").alias("match_method"),
    F.lit(1.0).alias("confidence")
)

matched_df = exact_matches_df.filter(F.col("sector_code").isNotNull())
unmatched_df = exact_matches_df.filter(F.col("sector_code").isNull()).drop(
    "sector_code", "sector_name", "taxonomy_type", "level", "parent_code", "match_method", "confidence"
)

matched_count = matched_df.count()
unmatched_count = unmatched_df.count()

print(f"Stage 1 - Exact Keyword Match Results:")
print(f"  Matched: {matched_count}")
print(f"  Unmatched: {unmatched_count}")

if matched_count > 0:
    print("\nExact keyword matches:")
    display(matched_df.orderBy("extraction_id"))

In [0]:
# Partial keyword matching using contains
partial_matches_df = unmatched_df.crossJoin(
    taxonomy_keywords_df.select(
        "sector_code", "sector_name", "taxonomy_type", "level", "parent_code", "normalized_keyword"
    ).distinct()
).filter(
    (F.col("normalized_sector").contains(F.col("normalized_keyword"))) |
    (F.col("normalized_keyword").contains(F.col("normalized_sector")))
).select(
    "extraction_id",
    "extracted_sector",
    "source_id",
    "normalized_sector",
    "batch_id",
    "source_name",
    "sector_code",
    "sector_name",
    "taxonomy_type",
    "level",
    "parent_code",
    F.lit("partial_keyword").alias("match_method"),
    F.lit(0.85).alias("confidence")
)

# Keep best match (most specific = highest level number)
if partial_matches_df.count() > 0:
    window_spec = Window.partitionBy("extraction_id").orderBy(F.desc("level"), F.desc("confidence"))
    
    partial_best_df = partial_matches_df.withColumn(
        "rank",
        F.row_number().over(window_spec)
    ).filter(
        F.col("rank") == 1
    ).drop("rank")
    
    partial_matched_count = partial_best_df.count()
    partial_unmatched_count = unmatched_count - partial_matched_count
    
    print(f"\nStage 2 - Partial Keyword Match Results:")
    print(f"  Matched: {partial_matched_count}")
    print(f"  Unmatched: {partial_unmatched_count}")
    
    if partial_matched_count > 0:
        print("\nPartial keyword matches:")
        display(partial_best_df)
    
    # Get remaining unmatched
    final_unmatched_df = unmatched_df.join(
        partial_best_df.select("extraction_id"),
        "extraction_id",
        "left_anti"
    )
    
    # Combine all matches
    all_matched_df = matched_df.union(partial_best_df)
else:
    print(f"\nStage 2 - No partial matches found")
    final_unmatched_df = unmatched_df
    all_matched_df = matched_df

In [0]:
# Optionally roll up to parent categories for more general classification
if CONFIG["enable_hierarchical_rollup"]:
    print(f"\nApplying hierarchical rollup to level {CONFIG['max_taxonomy_level']}...")
    
    # For matches that are too detailed, roll up to parent
    rollup_df = all_matched_df.filter(
        F.col("level") > CONFIG["max_taxonomy_level"]
    )
    
    if rollup_df.count() > 0:
        # Join with parent codes to get rolled-up sectors
        rolled_up_df = rollup_df.join(
            sector_taxonomy_df.select(
                F.col("sector_code").alias("parent_sector_code"),
                F.col("sector_name").alias("parent_sector_name"),
                F.col("level").alias("parent_level")
            ),
            rollup_df.parent_code == F.col("parent_sector_code"),
            "left"
        ).select(
            "extraction_id",
            "extracted_sector",
            "source_id",
            "normalized_sector",
            F.col("parent_sector_code").alias("sector_code"),
            F.col("parent_sector_name").alias("sector_name"),
            F.col("taxonomy_type"),
            F.col("parent_level").alias("level"),
            F.lit(None).alias("parent_code"),
            F.lit("hierarchical_rollup").alias("match_method"),
            F.col("confidence") * 0.9
        )
        
        # Replace rolled-up records in matched set
        keep_df = all_matched_df.filter(F.col("level") <= CONFIG["max_taxonomy_level"])
        all_matched_df = keep_df.union(rolled_up_df)
        
        print(f"Rolled up {rollup_df.count()} records to parent categories")
else:
    print("\nHierarchical rollup disabled")

In [0]:
# Enrich matched sectors with hierarchy information for warehouse consumption
# Create sector hierarchy mapping (category, level, parent relationships)

# Build sector hierarchy reference
sector_hierarchy = [
    # (canonical_name, category, level, parent)
    ("Information", "Technology", 1, None),
    ("Professional, Scientific, and Technical Services", "Professional Services", 1, None),
    ("Health Care and Social Assistance", "Healthcare", 1, None),
    ("Finance and Insurance", "Finance", 1, None),
    ("Retail Trade", "Retail", 1, None),
    ("Manufacturing", "Manufacturing", 1, None),
    ("Educational Services", "Education", 1, None),
    ("Software Publishers", "Technology", 2, "Information"),
    ("Computer Systems Design", "Technology", 2, "Information"),
]

hierarchy_df = spark.createDataFrame(
    sector_hierarchy,
    ["sector_name", "sector_category", "sector_level", "parent_sector"]
)

print("Sector hierarchy reference built")
print(f"Hierarchical mappings: {hierarchy_df.count()}")

# Enrich matched sectors with hierarchy (preserve batch fields)
enriched_df = all_matched_df.join(
    hierarchy_df,
    all_matched_df.sector_name == hierarchy_df.sector_name,
    "left"
).select(
    all_matched_df["extraction_id"],
    all_matched_df["extracted_sector"],
    all_matched_df["source_id"],
    all_matched_df["normalized_sector"],
    all_matched_df["batch_id"],
    all_matched_df["source_name"],
    all_matched_df["sector_code"],
    all_matched_df["sector_name"],
    all_matched_df["taxonomy_type"],
    all_matched_df["level"],
    all_matched_df["parent_code"],
    all_matched_df["match_method"],
    all_matched_df["confidence"],
    hierarchy_df["sector_category"],
    F.coalesce(hierarchy_df["sector_level"], F.lit(1)).alias("sector_level_hierarchy"),
    hierarchy_df["parent_sector"]
)

print("Sectors enriched with hierarchy information")
display(enriched_df.limit(10))

In [0]:
# Split by confidence threshold
high_confidence_df = enriched_df.filter(
    F.col("confidence") >= CONFIG["confidence_threshold"]
).withColumn(
    "processed_timestamp",
    F.current_timestamp()
)

low_confidence_df = enriched_df.filter(
    F.col("confidence") < CONFIG["confidence_threshold"]
).withColumn(
    "review_reason",
    F.lit("low_confidence")
).withColumn(
    "review_status",
    F.lit("pending")
).withColumn(
    "processed_timestamp",
    F.current_timestamp()
)

# Add unmatched to review queue
unmatched_review_df = final_unmatched_df.select(
    "extraction_id",
    "extracted_sector",
    "source_id",
    "normalized_sector",
    "batch_id",
    "source_name",
    F.lit(None).cast(StringType()).alias("sector_code"),
    F.lit(None).cast(StringType()).alias("sector_name"),
    F.lit(CONFIG["primary_taxonomy"]).alias("taxonomy_type"),
    F.lit(None).cast(IntegerType()).alias("level"),
    F.lit(None).cast(StringType()).alias("parent_code"),
    F.lit("no_match").alias("match_method"),
    F.lit(0.0).alias("confidence"),
    F.lit(None).cast(StringType()).alias("sector_category"),
    F.lit(None).cast(IntegerType()).alias("sector_level_hierarchy"),
    F.lit(None).cast(StringType()).alias("parent_sector"),
    F.lit("no_match").alias("review_reason"),
    F.lit("pending").alias("review_status"),
    F.current_timestamp().alias("processed_timestamp")
)

review_queue_df = low_confidence_df.union(unmatched_review_df)

high_conf_count = high_confidence_df.count()
review_count = review_queue_df.count()

print(f"\nNormalization Summary:")
print(f"  High confidence (>= {CONFIG['confidence_threshold']}): {high_conf_count}")
print(f"  Review queue: {review_count}")
print(f"  Success rate: {high_conf_count / extracted_count * 100:.1f}%")

In [0]:
# Prepare data for sem_sector_map schema
from pyspark.sql.functions import md5, concat, lit, when

output_df = high_confidence_df.select(
    md5(concat(F.col("extraction_id"), F.col("sector_code"))).alias("sector_map_id"),
    F.col("extraction_id").alias("enterprise_job_id"),
    F.col("sector_code").alias("canonical_sector_code"),
    F.col("sector_name").alias("canonical_sector_name"),  # Normalized NAICS sector
    F.lit("NAICS").alias("taxonomy_type"),
    when(F.col("match_method") == "exact_keyword", "EXACT_MATCH")
     .when(F.col("match_method") == "partial_keyword", "FUZZY")
     .when(F.col("match_method") == "hierarchical_rollup", "ROLLUP")
     .otherwise("MANUAL").alias("normalization_method"),
    F.col("confidence").cast("decimal(5,4)").alias("normalization_confidence"),
    F.to_json(F.struct(
        F.col("extracted_sector").alias("input"),
        F.col("normalized_sector").alias("normalized"),
        F.col("match_method").alias("method"),
        F.col("level").alias("taxonomy_level")
    )).alias("explanation_json"),
    F.col("batch_id").alias("effective_batch_id")
)

# Write to semantic.sem_sector_map using MERGE (idempotent)
print(f"\nWriting {high_conf_count} sector mappings to {CONFIG['output_table']}...")

# Create temp view for merge
output_df.createOrReplaceTempView("sector_map_updates")

spark.sql(f"""
MERGE INTO {CONFIG['output_table']} target
USING sector_map_updates source
ON target.sector_map_id = source.sector_map_id
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *
""")

print("✓ Sector mappings written successfully (idempotent MERGE)")

# Write review queue
if review_count > 0:
    print(f"\nWriting {review_count} records to review queue at {CONFIG['review_queue_table']}...")
    
    review_df = review_queue_df.select(
        F.expr("uuid()").alias("review_id"),
        F.col("extraction_id").alias("enterprise_job_id"),
        F.col("review_reason").alias("issue_type"),
        F.concat(
            F.lit("Sector: "),
            F.col("extracted_sector"),
            F.lit(", Normalized: "),
            F.col("normalized_sector"),
            F.lit(", Suggested: "),
            F.coalesce(F.col("sector_name"), F.lit("none"))
        ).alias("issue_detail"),
        F.coalesce(F.col("confidence"), F.lit(0.0)).cast("decimal(5,4)").alias("confidence"),
        F.current_timestamp().alias("queued_at"),
        F.col("review_status").alias("status"),
        F.col("batch_id"),
        F.col("source_name")
    )
    
    review_df.write \
        .format("delta") \
        .mode("append") \
        .option("mergeSchema", "true") \
        .saveAsTable(CONFIG["review_queue_table"])
    
    print("✓ Review queue records written successfully")

# Record batch processing metadata
print(f"\nRecording batch metadata to {metadata_table}...")

for batch_id_val, source_name_val in batches_to_process:
    # Get counts for this specific batch
    batch_canonical_count = high_confidence_df.filter(
        (F.col("batch_id") == batch_id_val) & (F.col("source_name") == source_name_val)
    ).count()
    
    batch_review_count = review_queue_df.filter(
        (F.col("batch_id") == batch_id_val) & (F.col("source_name") == source_name_val)
    ).count()
    
    batch_total = batch_canonical_count + batch_review_count
    
    metadata_record = spark.createDataFrame(
        [(
            batch_id_val,
            source_name_val,
            batch_total,
            batch_canonical_count,
            batch_review_count,
            run_timestamp_py,
            run_id,
            "completed"
        )],
        metadata_schema
    )
    
    metadata_record.write.format("delta").mode("append").saveAsTable(metadata_table)
    print(f"  ✓ Batch {batch_id_val} ({source_name_val}): {batch_total} sectors, {batch_canonical_count} mapped, {batch_review_count} for review")

print("\n✓ Batch metadata recorded successfully")

In [0]:
# Analyze sector distribution by taxonomy level
level_dist_df = high_confidence_df.groupBy("level", "taxonomy_type").agg(
    F.count("*").alias("count")
).orderBy("level")

print("\nSector Distribution by Taxonomy Level:")
display(level_dist_df)

# Top sectors
top_sectors_df = high_confidence_df.groupBy("sector_code", "sector_name").agg(
    F.count("*").alias("count")
).orderBy(F.desc("count")).limit(10)

print("\nTop 10 Normalized Sectors:")
display(top_sectors_df)

# Match method breakdown
method_stats_df = high_confidence_df.groupBy("match_method").agg(
    F.count("*").alias("count"),
    F.avg("confidence").alias("avg_confidence")
).orderBy(F.desc("count"))

print("\nMatch Method Statistics:")
display(method_stats_df)

print("\n" + "="*60)
print("SECTOR NORMALIZATION - EXECUTION SUMMARY")
print("="*60)
print(f"Input sectors: {extracted_count}")
print(f"Successfully normalized: {high_conf_count} ({high_conf_count/extracted_count*100:.1f}%)")
print(f"Review queue: {review_count} ({review_count/extracted_count*100:.1f}%)")
print(f"Primary taxonomy: {CONFIG['primary_taxonomy']}")
print(f"Confidence threshold: {CONFIG['confidence_threshold']}")
print("="*60)

In [0]:
import json

# Build summary for orchestration
summary = {
    "status": "completed",
    "run_id": run_id,
    "batches_processed": len(batches_to_process),
    "batch_ids": [b[0] for b in batches_to_process],
    "total_sectors": extracted_count,
    "canonical_count": high_conf_count,
    "review_queue_count": review_count,
    "success_rate": round(high_conf_count / extracted_count * 100, 2) if extracted_count > 0 else 0,
    "confidence_threshold": CONFIG["confidence_threshold"],
    "output_table": CONFIG["output_table"],
    "metadata_table": metadata_table
}

print("\n" + "="*60)
print("SECTOR NORMALIZATION - BATCH SUMMARY")
print("="*60)
print(json.dumps(summary, indent=2))
print("="*60)

dbutils.notebook.exit(json.dumps(summary))